In [ ]:
import multiprocessing
multiprocessing.set_start_method('fork', force=True)

Part 1: Heart Disease

In [ ]:
import pandas as pd

DATA_FILE = "cleveland.csv"

cleveland_data = pd.read_csv(DATA_FILE)

cleveland_data.head()

In [ ]:
columns = cleveland_data.columns.tolist()
columns.remove('num')  # Remove the "num" column for the features list to choose from for target variables
print(columns)

In [ ]:
# Give a heart disease true or false column based on the "num" column
cleveland_data['heart_disease'] = cleveland_data['num'] > 0
cleveland_data.head()

In [ ]:
# Generate a few different permutations of headings to test k-nearest neighbors on, and see which one performs the best.
from itertools import combinations
from math import comb
import random


def generate_subsets(colums, n, attr, seed=None):
    # Remove duplicate values from the input while preserving their order.
    values = list(dict.fromkeys(colums))

    if not 0 <= attr <= len(values):
        raise ValueError("k must be between 0 and the number of unique values")

    total = comb(len(values), attr)
    if not 0 <= n <= total:
        raise ValueError(f"n must be between 0 and {total}")

    rng = random.Random(seed)
    subsets = list(combinations(values, attr))
    yield from rng.sample(subsets, n)

In [ ]:
sets = list()
for subset in generate_subsets(columns, n=100, attr=4, seed=None):
    sets.append(subset)
    print(sets[-1])

In [ ]:
from collections.abc import Sequence
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Each process gets its own copy of these values through the initializer.
_WORKER_DATA = None
_WORKER_Y = None
_WORKER_SPLITS = None
_WORKER_NEIGHBOR_VALUES = None


def _predict_knn(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    n_neighbors: int,
) -> np.ndarray:
    """Predict binary labels using Euclidean-distance KNN."""
    distances = np.sum(
        (X_test[:, None, :] - X_train[None, :, :]) ** 2,
        axis=2,
    )
    nearest = np.argpartition(
        distances,
        kth=n_neighbors - 1,
        axis=1,
    )[:, :n_neighbors]

    predictions = []
    for neighbor_indices in nearest:
        votes = np.bincount(
            y_train[neighbor_indices],
            minlength=2,
        )
        # A tie is classified as 0 because np.argmax chooses the first max.
        predictions.append(int(np.argmax(votes)))

    return np.asarray(predictions)


def _init_knn_worker(data, y, split_indices, neighbor_values):
    """Initialize data shared by tasks running in each process."""
    global _WORKER_DATA
    global _WORKER_Y
    global _WORKER_SPLITS
    global _WORKER_NEIGHBOR_VALUES

    _WORKER_DATA = data
    _WORKER_Y = y
    _WORKER_SPLITS = split_indices
    _WORKER_NEIGHBOR_VALUES = neighbor_values


def _evaluate_subset_split(task):
    """Evaluate one feature subset on one train/test split."""
    subset, split_index = task
    split_seed, train_ids, test_ids = _WORKER_SPLITS[split_index]

    X = (
        _WORKER_DATA.loc[:, list(subset)]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    X_train = X[train_ids]
    X_test = X[test_ids]
    y_train = _WORKER_Y[train_ids]
    y_test = _WORKER_Y[test_ids]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    rows = []

    for n_neighbors in _WORKER_NEIGHBOR_VALUES:
        predictions = _predict_knn(
            X_train,
            y_train,
            X_test,
            n_neighbors,
        )

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1],
        ).ravel()

        rows.append(
            {
                "attributes": subset,
                "n_neighbors": n_neighbors,
                "split_seed": split_seed,
                "precision": precision_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "recall": recall_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "f1": f1_score(
                    y_test,
                    predictions,
                    zero_division=0,
                ),
                "tn": tn,
                "fp": fp,
                "fn": fn,
                "tp": tp,
            }
        )

    return rows


def run_knn_on_subsets(
    data: pd.DataFrame,
    subsets: Sequence[tuple[str, ...]],
    target_column: str = "num",
    test_size: float = 0.2,
    seed: int | None = None,
    iterations: int = 10,
    neighbor_values: Sequence[int] = (1, 3, 5, 7, 9, 11, 13, 15),
    processes: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run repeated train/test evaluations in parallel.

    processes=None uses ProcessPoolExecutor's default worker count.

    Returns:
        runs: One row per model per split, with split seed, metrics,
              and confusion-matrix counts.
        summary: Mean precision, recall, and F1, plus summed
                 confusion-matrix counts for each model.
    """
    if iterations < 1:
        raise ValueError("iterations must be at least 1")
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")
    if not subsets:
        raise ValueError("subsets must contain at least one feature subset")
    if not neighbor_values or any(k < 1 for k in neighbor_values):
        raise ValueError("neighbor_values must contain positive integers")
    if processes is not None and processes < 1:
        raise ValueError("processes must be at least 1 or None")

    master_seed = (
        int(seed)
        if seed is not None
        else int(np.random.SeedSequence().generate_state(1)[0])
    )
    rng = np.random.default_rng(master_seed)
    split_seeds = rng.integers(
        0,
        np.iinfo(np.int32).max,
        size=iterations,
    )

    # Convert Cleveland's num values to binary: 0 versus greater than 0.
    target = pd.to_numeric(data[target_column], errors="coerce")
    valid_rows = target.notna()
    y = (target.loc[valid_rows] > 0).astype(int).to_numpy()
    data = data.loc[valid_rows]

    if set(np.unique(y)) != {0, 1}:
        raise ValueError(
            "The usable target data must contain both classes 0 and 1"
        )

    normalized_subsets = []
    for subset in subsets:
        subset = tuple(subset)

        if not subset:
            raise ValueError("Feature subsets cannot be empty")
        if len(set(subset)) != len(subset):
            raise ValueError(f"Subset contains duplicate columns: {subset}")
        if target_column in subset:
            raise ValueError("The target column cannot also be a feature")

        missing = set(subset) - set(data.columns)
        if missing:
            raise ValueError(
                f"Columns not found in data: {sorted(missing)}"
            )

        normalized_subsets.append(subset)

    row_ids = np.arange(len(data))
    split_indices = []

    # Generate each split once and reuse it for every model.
    for split_seed in split_seeds:
        train_ids, test_ids = train_test_split(
            row_ids,
            test_size=test_size,
            random_state=int(split_seed),
            stratify=y,
        )
        split_indices.append(
            (int(split_seed), train_ids, test_ids)
        )

    largest_train_set = min(
        len(train_ids)
        for _, train_ids, _ in split_indices
    )
    if any(k > largest_train_set for k in neighbor_values):
        raise ValueError(
            "Every n_neighbors value must be no larger than the "
            f"smallest training set ({largest_train_set} rows)"
        )

    tasks = [
        (subset, split_index)
        for subset in normalized_subsets
        for split_index in range(iterations)
    ]

    rows = []
    with ProcessPoolExecutor(
        max_workers=processes,
        initializer=_init_knn_worker,
        initargs=(
            data,
            y,
            split_indices,
            tuple(neighbor_values),
        ),
    ) as executor:
        for task_rows in executor.map(_evaluate_subset_split, tasks):
            rows.extend(task_rows)

    runs = pd.DataFrame(rows)
    runs.attrs["master_seed"] = master_seed

    summary = (
        runs.groupby(
            ["attributes", "n_neighbors"],
            as_index=False,
            sort=False,
        )
        .agg(
            mean_precision=("precision", "mean"),
            mean_recall=("recall", "mean"),
            mean_f1=("f1", "mean"),
            tn=("tn", "sum"),
            fp=("fp", "sum"),
            fn=("fn", "sum"),
            tp=("tp", "sum"),
        )
        .sort_values("mean_f1", ascending=False)
        .reset_index(drop=True)
    )

    return runs, summary

In [ ]:
from collections import Counter
from itertools import combinations

feature_columns = [
    column for column in cleveland_data.columns
    if column not in ("num", "heart_disease")
]

neighbor_values = (1, 3, 5, 7, 9, 11)
iterations = 10
test_size = 0.2

subsets = [
    subset
    for size in range(3, 6)
    for subset in combinations(feature_columns, size)
]

runs, summary = run_knn_on_subsets(
    data=cleveland_data,
    subsets=subsets,
    target_column="heart_disease",
    test_size=test_size,
    seed=None,
    iterations=iterations,
    neighbor_values=neighbor_values,
)

# Count feature subsets by their number of attributes.
subset_counts = Counter(len(subset) for subset in subsets)

unique_configurations = len(subsets) * len(neighbor_values)
expected_evaluations = unique_configurations * iterations

report_lines = [
    "KNN HEART DISEASE CLASSIFICATION: EXPERIMENT SUMMARY",
    "",
    f"Dataset rows: {len(cleveland_data):,}",
    f"Target: heart_disease",
    f"Predictor attributes: {len(feature_columns)}",
    f"Predictors used: {', '.join(feature_columns)}",
    f"Test size: {test_size:.0%}",
    "",
    f"Feature subsets generated: {len(subsets):,}",
    "Feature subsets by size:",
]

for size in sorted(subset_counts):
    count = subset_counts[size]
    evaluations = count * len(neighbor_values) * iterations
    report_lines.append(
        f"  {size} attributes: {count:,} subsets, "
        f"{evaluations:,} model evaluations"
    )

report_lines.extend([
    "",
    f"Neighbor values tested: {', '.join(map(str, neighbor_values))}",
    f"Iterations per subset-and-neighbor configuration: {iterations}",
    f"Unique subset-and-neighbor configurations: {unique_configurations:,}",
    f"Expected model evaluations: {expected_evaluations:,}",
    f"Model evaluations recorded in runs: {len(runs):,}",
    f"Summary rows: {len(summary):,}",
    f"Master seed: {runs.attrs.get('master_seed', 'not recorded')}",
    "",
    "Target class counts:",
])

class_counts = cleveland_data["heart_disease"].value_counts(dropna=False)
for class_label, count in class_counts.items():
    report_lines.append(f"  {class_label}: {count:,}")

print("\n".join(report_lines))

In [ ]:
# Get the highest-ranked configuration from the generated summary.
best_row = summary.loc[summary["mean_f1"].idxmax()]
best_attributes = tuple(best_row["attributes"])
best_k = best_row["n_neighbors"]

# Select the individual runs for that configuration.
best_model_runs = runs[
    (runs["attributes"].map(tuple) == best_attributes)
    & (runs["n_neighbors"] == best_k)
]

print(
    f"Runs for the best model: attributes={best_attributes}, "
    f"k={best_k}, mean F1={best_row['mean_f1']:.3f}"
)
print(f"Number of runs: {len(best_model_runs)}")

display(best_model_runs[["precision", "recall", "f1"]])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

sns.set_theme(
    style="whitegrid",
    context="talk",
    palette="colorblind",
)

plot_summary = summary.copy()
plot_summary["n_attributes"] = plot_summary["attributes"].map(len)
plot_summary = plot_summary.sort_values("mean_f1", ascending=False)

top10 = plot_summary.head(10).copy()
top5 = plot_summary.head(5).copy()

# 1. Top 10 feature-set and neighbor-count combinations
fig, ax = plt.subplots(figsize=(14, 9))

labels = [
    textwrap.fill(
        f"{', '.join(attributes)} | k={neighbors}",
        width=38,
    )
    for attributes, neighbors in zip(
        top10["attributes"],
        top10["n_neighbors"],
    )
]
colors = sns.color_palette("viridis", n_colors=len(top10))

bars = ax.barh(labels, top10["mean_f1"], color=colors)
ax.invert_yaxis()
ax.set_title("Top 10 Configurations by Mean F1", fontweight="bold")
ax.set_xlabel("Mean F1")
ax.set_ylabel("")
ax.set_xlim(0, 1)

for bar, score in zip(bars, top10["mean_f1"]):
    ax.text(
        score + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.3f}",
        va="center",
        fontsize=11,
    )

fig.tight_layout()
fig.savefig("top_10_configurations.png", dpi=300, bbox_inches="tight")
plt.show()


# 2. Precision, recall, and F1 for the top 5 configurations
fig, ax = plt.subplots(figsize=(16, 8))

positions = range(len(top5))
bar_width = 0.25

metrics = [
    ("mean_precision", "Precision", "#4C78A8"),
    ("mean_recall", "Recall", "#F58518"),
    ("mean_f1", "F1", "#54A24B"),
]

for offset, (column, label, color) in enumerate(metrics):
    values = top5[column].to_numpy()
    x_positions = [
        position + (offset - 1) * bar_width
        for position in positions
    ]

    bars = ax.bar(
        x_positions,
        values,
        width=bar_width,
        label=label,
        color=color,
    )

    ax.bar_label(
        bars,
        fmt="%.3f",
        label_type="center",
        color="white",
        fontsize=9,
        fontweight="bold",
    )

configuration_labels = [
    textwrap.fill(
        f"{', '.join(attributes)}\nk={neighbors}",
        width=24,
    )
    for attributes, neighbors in zip(
        top5["attributes"],
        top5["n_neighbors"],
    )
]

ax.set_title("Metrics for the Top 5 Configurations", fontweight="bold")
ax.set_xlabel("Attributes and neighbor count")
ax.set_ylabel("Score")
ax.set_xticks(list(positions))
ax.set_xticklabels(configuration_labels, fontsize=10)
ax.set_ylim(0, 1)
ax.legend(title="Metric")

fig.tight_layout()
fig.savefig("top_5_metrics.png", dpi=300, bbox_inches="tight")
plt.show()


# 3. Mean F1 by number of attributes and neighbor count
fig, ax = plt.subplots(figsize=(12, 7))

grouped = (
    plot_summary
    .groupby(["n_attributes", "n_neighbors"], as_index=False)["mean_f1"]
    .mean()
)

for neighbor_count, group in grouped.groupby("n_neighbors"):
    group = group.sort_values("n_attributes")
    ax.plot(
        group["n_attributes"],
        group["mean_f1"],
        marker="o",
        linewidth=2,
        label=f"k={neighbor_count}",
    )

ax.set_title(
    "Mean F1 by Subset Size and Neighbor Count",
    fontweight="bold",
)
ax.set_xlabel("Number of attributes")
ax.set_ylabel("Mean F1")
ax.set_xticks([3, 4, 5])
ax.set_ylim(0, 1)
ax.legend(title="Neighbors", ncol=2, fontsize=10)

fig.tight_layout()
fig.savefig("mean_f1_by_subset_size.png", dpi=300, bbox_inches="tight")
plt.show()


# 4. Confusion matrix for the best configuration
best = plot_summary.iloc[0]
confusion_matrix = [
    [best["tn"], best["fp"]],
    [best["fn"], best["tp"]],
]

best_attributes = textwrap.fill(
    ", ".join(best["attributes"]),
    width=42,
)

fig, ax = plt.subplots(figsize=(9, 8))

sns.heatmap(
    confusion_matrix,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    cbar=False,
    square=True,
    linewidths=1,
    linecolor="white",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    ax=ax,
)

ax.set_title(
    "Best Configuration: Confusion Matrix\n"
    f"Attributes: {best_attributes}\n"
    f"k={best['n_neighbors']}, mean F1={best['mean_f1']:.3f}",
    fontweight="bold",
    fontsize=16,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")

fig.tight_layout()
fig.savefig("best_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

### Test Dataset & Challenge

Below is code to take the best found F1 score model and predict on a challenge set which people have heart disease and which people do not.

From our results, the best model was composed of the attributes (age, cp, exang, ca, thal) with k=5.

In [ ]:
from sklearn.metrics import confusion_matrix as sklearn_confusion_matrix

# The challenge patients are queried against the original Cleveland dataset.
TEST_SET = "cleveland-test-sample.csv"
BEST_ATTRIBUTES = ("age", "cp", "exang", "ca", "thal")
BEST_K = 5

challenge_data = pd.read_csv(TEST_SET)

# Use every usable row in the original dataset as a possible neighbor.
original_target = pd.to_numeric(
    cleveland_data["num"],
    errors="coerce",
)
original_valid = original_target.notna()
original_features = (
    cleveland_data.loc[original_valid, list(BEST_ATTRIBUTES)]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float)
)
original_labels = (original_target.loc[original_valid] > 0).astype(int).to_numpy()

challenge_features = (
    challenge_data.loc[:, list(BEST_ATTRIBUTES)]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float)
)
challenge_labels = pd.to_numeric(
    challenge_data["disease"],
    errors="coerce",
).to_numpy(dtype=int)

# Learn preprocessing parameters from the original data only.
imputer = SimpleImputer(strategy="median")
original_features = imputer.fit_transform(original_features)
challenge_features = imputer.transform(challenge_features)

scaler = StandardScaler()
original_features = scaler.fit_transform(original_features)
challenge_features = scaler.transform(challenge_features)

challenge_predictions = _predict_knn(
    original_features,
    original_labels,
    challenge_features,
    BEST_K,
)

challenge_f1 = f1_score(
    challenge_labels,
    challenge_predictions,
    zero_division=0,
)
challenge_confusion = sklearn_confusion_matrix(
    challenge_labels,
    challenge_predictions,
    labels=[0, 1],
)

challenge_results = challenge_data.copy()
challenge_results["prediction"] = challenge_predictions

print(f"Attributes: {BEST_ATTRIBUTES}")
print(f"Neighbors: {BEST_K}")
print(f"Challenge F1 score: {challenge_f1:.3f}")
print("Confusion matrix [rows=actual, columns=predicted]:")
print(challenge_confusion)
display(challenge_results[["disease", "prediction"]])

## Part 2

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import KNeighborsRegressor
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv('boardgames_ranks.csv')
df.head()

In [ ]:
baseGames = df[df['is_expansion'] == 0]
# 1. Define our features (X) and target variable (y)
feature_columns = [col for col in baseGames.columns if col.endswith('_rank')]
# Fill missing ranks with a high number to indicate they are unranked in that category
X = baseGames[feature_columns].fillna(10000000)
y = baseGames['rank']

# 2. Initialize and fit the model with exactly 10 neighbors
knn_reg = KNeighborsRegressor(n_neighbors=10, metric='euclidean')
knn_reg.fit(X, y)

# 3. Define the estimation function
def estimate_game_rank(game_name):
    # Find the index of the requested game
    if game_name not in baseGames['name'].values:
        return "Game not found in dataset."
        
    game_idx = baseGames[baseGames['name'] == game_name].index[0]
    
    # Extract the category rank features for this specific game
    game_features = X.loc[[game_idx]]
    
    # Predict the overall rank using the trained model
    estimated_rank = knn_reg.predict(game_features)[0]
    actual_rank = baseGames.loc[game_idx, 'rank']
    
    return {
        'Game': game_name,
        'Estimated Rank': int(round(estimated_rank)),
        'Actual Rank': actual_rank
    }

# Example usage
print(estimate_game_rank('Brass: Birmingham'))

In [ ]:
# 1. Split your data into training and testing sets so you can evaluate performance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

error_rates = []
# Test fewer K values, stepping by 2 (e.g., 1, 3, 5, 7...) to cut the workload in half
k_values = range(1, 30, 2) 

for k in k_values:
    # n_jobs=-1 tells your computer to use all CPU cores at once
    knn = KNeighborsRegressor(n_neighbors=k, metric='euclidean', n_jobs=-1)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    
    mse = mean_squared_error(y_test, pred_i)
    error_rates.append(mse)
    
    # Optional: Add a print statement so you can see it working!
    print(f"Finished K={k}")

# 3. Plot the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rates, color='blue', linestyle='dashed', marker='o', markerfacecolor='red', markersize=6)
plt.title('Elbow Curve: Mean Squared Error vs. K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Squared Error (MSE)')
plt.grid(True)
plt.show()

In [ ]:
# --- Part 2: Final Optimized Model ---

optimal_k = 13
print(f"Training optimized KNN model with K={optimal_k}...\n")

# 1. Initialize the model with our optimal K
final_knn = KNeighborsRegressor(n_neighbors=optimal_k, metric='euclidean', n_jobs=-1)

# 2. Fit on the training data first to show the final Test Error
final_knn.fit(X_train, y_train)
test_predictions = final_knn.predict(X_test)
final_mse = mean_squared_error(y_test, test_predictions)
print(f"Performance Metrics for K={optimal_k}:")
print(f"Mean Squared Error (MSE): {final_mse:.2f}")

# 3. Retrain on the ENTIRE dataset (X, y) so our estimator has maximum data to pull from
final_knn.fit(X, y)

# 4. Updated estimation function using the new final_knn model
def estimate_game_rank_optimized(game_name):
    if game_name not in baseGames['name'].values:
        return "Game not found in dataset."
        
    # Find index and extract features
    game_idx = baseGames[baseGames['name'] == game_name].index[0]
    game_features = X.loc[[game_idx]]
    
    # Predict with the k=13 model
    estimated_rank = final_knn.predict(game_features)[0]
    actual_rank = baseGames.loc[game_idx, 'rank']
    
    return {
        'Game': game_name, 
        'Estimated Rank': int(round(estimated_rank)), 
        'Actual Rank': actual_rank
    }

# Let's test it out with our new optimized model!
print("\nPrediction using Optimized Model:")
print(estimate_game_rank_optimized('Brass: Birmingham'))
print(estimate_game_rank_optimized('Agricola'))
print(estimate_game_rank_optimized('Wingspan'))
print(estimate_game_rank_optimized('Cthulhu: Death May Die'))

In [ ]:
import random

# 1. The specific games you want to highlight
target_games = [
    'Brass: Birmingham', 
    'Agricola', 
    'Cthulhu: Death May Die',
    'Clank!: Catacombs',
    'Obsession',
    'The Crew: Mission Deep Sea',
    'Harmonies',
    'Pandemic Legacy: Season 2',
    'Pandemic',
    'Flip 7',
    'Wingspan'
]

# 2. Grab 5 random games from the top 150 (excluding the ones we already picked)
top_150 = baseGames[(baseGames['rank'] <= 150) & (baseGames['rank'] > 0) & (~baseGames['name'].isin(target_games))]
random_top_games = random.sample(top_150['name'].tolist(), 5)

# Combine our lists
games_to_test = target_games # + random_top_games

# 3. Loop through the games and collect the results
results_list = []
for game in games_to_test:
    res = estimate_game_rank_optimized(game)
    
    # Ensure the game was found before processing
    if isinstance(res, dict): 
        # Calculate how far off the prediction was
        res['Off By'] = abs(res['Estimated Rank'] - res['Actual Rank'])
        results_list.append(res)

# 4. Convert the list of dictionaries into a Pandas DataFrame
results_df = pd.DataFrame(results_list)

# Sort the table by the Actual Rank so it reads nicely top-to-bottom
results_df = results_df.sort_values('Actual Rank').reset_index(drop=True)

# 5. Display the beautiful table! (Using display() instead of print() in Jupyter makes it format as HTML)
display(results_df)